# Calibrate Intergrowth Hard/Normal Morphology

Calibrate a local-window morphology classifier that turns metallic ore masks into a parallel intergrowth mask: background, talc, normal ore, hard ore, and ignore. Talc remains manual/reviewed only; this notebook does not create talc from RGB/HSV thresholds.

In [ ]:
from pathlib import Path
import json
import sys

from PIL import Image
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ore_detection.data.inventory import inventory_baseline_images
from ore_detection.backend.ui_annotation import BASE_UI_CLASSES, class_index_to_color_image
from ore_detection.descriptors.intergrowth_classification import (
    IntergrowthClassifierConfig,
    choose_hard_threshold,
    classify_intergrowth_mask,
    save_intergrowth_classifier_config,
)

BASELINE_ROOT = PROJECT_ROOT / 'datasets' / 'baseline'
PREDICTION_ROOT = PROJECT_ROOT / 'data_work' / 'predictions' / 'model_segmentation' / 'baseline_crops'
CLASSIFIER_PATH = PROJECT_ROOT / 'models' / 'intergrowth_classifier' / '001' / 'classifier.json'

RUN_CALIBRATION = False
WINDOW_SIZE = 128
STRIDE = 64
SMALL_AREA_THRESHOLD = 25
DEFAULT_HARD_THRESHOLD = 0.5
EFFECTIVE_STRIDE = min(STRIDE, WINDOW_SIZE)

print(f'baseline root: {BASELINE_ROOT}')
print(f'classifier path: {CLASSIFIER_PATH}')
print(f'window size: {WINDOW_SIZE}; requested stride: {STRIDE}; effective stride: {EFFECTIVE_STRIDE}')

baseline root: C:\Users\Cape\PycharmProjects\ore_detection\datasets\baseline
classifier path: C:\Users\Cape\PycharmProjects\ore_detection\models\intergrowth_classifier\001\classifier.json
window size: 128; requested stride: 64; effective stride: 64


The first calibration pass uses weak crop labels from `Normal ore` and `Hard ore`. `Talc contained` crops are reported as diagnostics only because talc is a separate manual class and the metallic ore pixels still need reviewed normal/hard labels.

In [ ]:
baseline_records = inventory_baseline_images(BASELINE_ROOT)
baseline_records = sorted(baseline_records, key=lambda row: (row['part'], row['label'], row['path']))
label_counts = {}
for record in baseline_records:
    label_counts[str(record['label'])] = label_counts.get(str(record['label']), 0) + 1
print('baseline crop count (panoramas skipped):', len(baseline_records))
print(label_counts)

BASELINE_ROOT_RESOLVED = BASELINE_ROOT.resolve()

def is_baseline_crop_path(path):
    path = Path(path).resolve()
    try:
        rel = path.relative_to(BASELINE_ROOT_RESOLVED)
    except ValueError:
        return False
    return len(rel.parts) >= 3 and 'panoramas' not in {part.lower() for part in rel.parts}

def baseline_crop_key(path):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(BASELINE_ROOT_RESOLVED)).casefold()
    except ValueError:
        return str(path).casefold()

def expected_prediction_dir_for_record(record):
    sample_id = f"{record['part']}_{record['label']}_{Path(record['path']).stem}".replace(' ', '_')
    return PREDICTION_ROOT / sample_id

def prediction_dirs_by_image_path():
    matched = {}
    skipped_panorama_predictions = 0
    missing_or_bad_metadata = 0
    if not PREDICTION_ROOT.exists():
        return matched
    for sample_dir in sorted(path for path in PREDICTION_ROOT.glob('*') if path.is_dir() and (path / 'ore_mask.png').exists()):
        metadata_path = sample_dir / 'metadata.json'
        if not metadata_path.exists():
            missing_or_bad_metadata += 1
            continue
        try:
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            missing_or_bad_metadata += 1
            continue
        image_value = metadata.get('image_path')
        if not image_value:
            missing_or_bad_metadata += 1
            continue
        image_path = Path(image_value)
        image_path = image_path.resolve() if image_path.is_absolute() else (PROJECT_ROOT / image_path).resolve()
        if not is_baseline_crop_path(image_path):
            skipped_panorama_predictions += 1
            continue
        matched[baseline_crop_key(image_path)] = sample_dir
    print(f'prediction dirs matched by metadata: {len(matched)}')
    print(f'prediction dirs skipped as panoramas: {skipped_panorama_predictions}')
    print(f'prediction dirs skipped for missing/bad metadata: {missing_or_bad_metadata}')
    return matched

prediction_dir_by_image_key = prediction_dirs_by_image_path()

def prediction_dir_for_record(record):
    return prediction_dir_by_image_key.get(baseline_crop_key(record['path']), expected_prediction_dir_for_record(record))


For each prediction directory, load `ore_mask.png` and summarize local intergrowth scores. This notebook expects baseline crop predictions to already exist from `04_predict_ore_masks_and_descriptors.ipynb`; prediction generation stays disabled here by default.

In [ ]:
def image_score_from_prediction(prediction_dir, config):
    with Image.open(prediction_dir / 'ore_mask.png') as mask:
        result = classify_intergrowth_mask(mask, config=config)
    return result.metrics['hard_fraction_of_metallic_ore']

config = IntergrowthClassifierConfig(
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    small_area_threshold=SMALL_AREA_THRESHOLD,
    hard_threshold=DEFAULT_HARD_THRESHOLD,
)

labeled_scores = []
diagnostic_scores = []
matched_prediction_counts = {}
missing_prediction_counts = {}
if RUN_CALIBRATION:
    for record in baseline_records:
        prediction_dir = prediction_dir_for_record(record)
        label = str(record['label'])
        if not (prediction_dir / 'ore_mask.png').exists():
            missing_prediction_counts[label] = missing_prediction_counts.get(label, 0) + 1
            continue
        score = image_score_from_prediction(prediction_dir, config)
        matched_prediction_counts[label] = matched_prediction_counts.get(label, 0) + 1
        if label in {'Normal ore', 'Hard ore'}:
            labeled_scores.append((score, label))
        else:
            diagnostic_scores.append((score, label, record['path']))

    print('matched prediction counts:', matched_prediction_counts)
    print('missing prediction counts:', missing_prediction_counts)
    labels_with_scores = {label for _, label in labeled_scores}
    if not {'Normal ore', 'Hard ore'} <= labels_with_scores:
        print('Need both Normal ore and Hard ore prediction masks before saving a calibrated threshold.')
        print('Run notebook 04 with RUN_PREDICTION=True and SAMPLE_LIMIT=None to create all baseline crop predictions.')
        print('Classifier not saved; current config remains diagnostic only:')
        print(json.dumps(config.to_dict(), indent=2))
    else:
        threshold = choose_hard_threshold(labeled_scores)
        calibrated = IntergrowthClassifierConfig.from_dict({**config.to_dict(), 'hard_threshold': threshold['threshold']})
        save_intergrowth_classifier_config(calibrated, CLASSIFIER_PATH)
        print(json.dumps(threshold, indent=2))
        print(f'saved: {CLASSIFIER_PATH}')
else:
    print('Set RUN_CALIBRATION = True after baseline Normal ore and Hard ore predictions exist.')

## Visual Review

Review raw image, mineral mask, hard ore score, and normal ore score. This block does not save artifacts; it lets you check the local morphology classifier by eye before accepting thresholds.


In [ ]:
REVIEW_LIMIT = 5

def normal_score_from_hard_score(hard_score, ore_mask):
    hard_values = hard_score.convert('L').tobytes()
    ore_values = ore_mask.convert('L').tobytes()
    normal = Image.new('L', hard_score.size)
    normal.putdata([255 - int(score) if ore > 0 else 0 for score, ore in zip(hard_values, ore_values)])
    return normal

def load_mineral_preview(sample_dir, ore_mask):
    multiclass_path = sample_dir / 'ore_multiclass_mask.png'
    if multiclass_path.exists():
        return Image.open(multiclass_path).convert('L'), 'mineral mask'
    return ore_mask.convert('L').copy(), 'mineral mask (binary fallback)'

review_records = []
for record in baseline_records:
    prediction_dir = prediction_dir_for_record(record)
    if (prediction_dir / 'ore_mask.png').exists():
        review_records.append((record, prediction_dir))

print(f'prediction dirs available for visual review: {len(review_records)}')

for record, sample_dir in review_records[:REVIEW_LIMIT]:
    metadata_path = sample_dir / 'metadata.json'
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        image_path = Path(metadata.get('image_path', record['path']))
    else:
        image_path = Path(record['path'])

    with Image.open(image_path) as raw, Image.open(sample_dir / 'ore_mask.png') as mask:
        mask_l = mask.convert('L')
        result = classify_intergrowth_mask(mask_l, config=config)
        normal_score = normal_score_from_hard_score(result.score, mask_l)
        mineral_preview, mineral_title = load_mineral_preview(sample_dir, mask_l)
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(raw.convert('RGB'))
        axes[0].set_title('raw image')
        axes[1].imshow(mineral_preview, cmap='tab20')
        axes[1].set_title(mineral_title)
        axes[2].imshow(result.score, cmap='magma', vmin=0, vmax=255)
        axes[2].set_title('hard ore score')
        axes[3].imshow(normal_score, cmap='viridis', vmin=0, vmax=255)
        axes[3].set_title('normal ore score')
        fig.suptitle(
            f"{record['part']} / {record['label']} / {Path(record['path']).name} | "
            f"label={result.metrics['image_label']} | "
            f"hard={result.metrics['hard_fraction_of_metallic_ore']:.3f}"
        )
        for ax in axes:
            ax.axis('off')
        plt.show()
        mineral_preview.close()
